# Motor-imagery EEG channel-loss quickstart

This notebook runs the repository's real **CSP–LDA** pipeline on one participant
from the public PhysioNet EEG Motor Movement/Imagery dataset. The model is fit on
clean data and evaluated after deterministic test-time channel dropout.

**Scope:** this compact run is for trying the workflow. It does not reproduce or
support population-level claims from the full 109-participant analysis.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/ZyntZ/motor-imagery-eeg-decoder-robustness.git"
cwd = Path.cwd()
if (cwd / "pyproject.toml").exists():
    REPO = cwd
else:
    clone_parent = Path("/content") if Path("/content").exists() else cwd
    REPO = clone_parent / "motor-imagery-eeg-decoder-robustness"

if not (REPO / "pyproject.toml").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[eeg,reports]"],
    check=True,
)
os.chdir(REPO)
print(f"Repository: {REPO}")

## Run one participant

The quickstart evaluates 0%, 10%, 30%, and 50% dropout. Non-zero conditions use
two deterministic masks each. The first run downloads the public EEG files and
can take longer than later runs.

In [ ]:
command = [
    sys.executable,
    "scripts/run_benchmark.py",
    "--config", "configs/quickstart_physionet.yaml",
    "--download-and-run",
    "--dataset", "PhysionetMI",
    "--subjects", "1",
    "--pipeline", "csp_lda",
    "--overwrite",
    "--suffix", "quickstart",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

## Inspect the result

Each point below is the participant's mean across cross-validation folds and,
for non-zero dropout, the two deterministic channel masks. With only one
participant, no population confidence interval is estimated.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

summary_path = REPO / "results" / "quickstart" / "PhysionetMI_quickstart_subject_summary.csv"
summary = pd.read_csv(summary_path).sort_values("dropout_fraction")
view = summary[[
    "dropout_fraction",
    "n_channels",
    "n_dropped_channels",
    "roc_auc",
    "balanced_accuracy",
]].copy()
view["dropout_percent"] = (100 * view.pop("dropout_fraction")).round().astype(int)
view = view[["dropout_percent", "n_channels", "n_dropped_channels", "roc_auc", "balanced_accuracy"]]
display(view.round(3))

ax = view.plot(
    x="dropout_percent",
    y="roc_auc",
    marker="o",
    legend=False,
    figsize=(7, 4),
)
ax.axhline(0.5, color="0.5", linestyle="--", linewidth=1, label="chance")
ax.set(
    xlabel="Channels zeroed at test time (%)",
    ylabel="ROC-AUC",
    title="CSP–LDA robustness for PhysioNet participant 1",
    ylim=(0, 1),
)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Next steps

- Run additional subjects by changing `--subjects 1` to multiple IDs.
- Restore 10 masks and all configured dropout fractions with
  `configs/benchmark_independent_masks.yaml`.
- See `REPRODUCIBILITY.md` before interpreting or comparing population results.